<a href="https://colab.research.google.com/github/Aayush-Jamwal/prompt-engineering-week2/blob/master/Qwen_Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai transformers accelerate bitsandbytes

import json
import time

with open("prompts_english.json", "r", encoding="utf-8") as f:
    english_prompts = json.load(f)

with open("prompts_hindi.json", "r", encoding="utf-8") as f:
    hindi_prompts = json.load(f)

all_prompts = english_prompts + hindi_prompts
print("Total prompts loaded:", len(all_prompts))

SYSTEM_PROMPT = (
    "You are AgriVision, an agricultural expert. Answer the farmer's question clearly and concisely. "
    "If it is a crop disease, state the exact English disease name first, followed by the treatment."
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.2.0 which is incompatible.
Total prompts loaded: 60


LOOPING PROMPTS ON QWEN via vLLM

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen3-8B"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

qwen_results = []

for i in all_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": i["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150)
    latency_ms = (time.time() - start) * 1000

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    expected = i.get("expected_answer") or i.get("expected_disease") or ""
    is_correct = expected.lower() in output.lower()

    qwen_results.append({
        "id": i["id"],
        "correct": is_correct,
        "latency_ms": round(latency_ms, 1),
        "output": output
    })
    print(f"Qwen-8B | {i['id']} | {latency_ms:.0f} ms | Correct: {is_correct}")

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Qwen-8B | EN_01 | 10627 ms | Correct: False
Qwen-8B | EN_02 | 12123 ms | Correct: False
Qwen-8B | EN_03 | 14129 ms | Correct: False
Qwen-8B | EN_04 | 11502 ms | Correct: False
Qwen-8B | EN_05 | 14410 ms | Correct: True
Qwen-8B | EN_06 | 6470 ms | Correct: True
Qwen-8B | EN_07 | 6759 ms | Correct: False
Qwen-8B | EN_08 | 6621 ms | Correct: True
Qwen-8B | EN_09 | 15971 ms | Correct: False
Qwen-8B | EN_10 | 12342 ms | Correct: False
Qwen-8B | EN_11 | 8106 ms | Correct: True
Qwen-8B | EN_12 | 14210 ms | Correct: True
Qwen-8B | EN_13 | 7235 ms | Correct: True
Qwen-8B | EN_14 | 14858 ms | Correct: False
Qwen-8B | EN_15 | 8663 ms | Correct: True
Qwen-8B | EN_16 | 8161 ms | Correct: False
Qwen-8B | EN_17 | 8736 ms | Correct: True
Qwen-8B | EN_18 | 11796 ms | Correct: False
Qwen-8B | EN_19 | 11502 ms | Correct: False
Qwen-8B | EN_20 | 9364 ms | Correct: False
Qwen-8B | EN_21 | 10267 ms | Correct: False
Qwen-8B | EN_22 | 8753 ms | Correct: True
Qwen-8B | EN_23 | 8836 ms | Correct: True
Qwen-8B |

CALCS

In [ ]:
# Gemini Summary
g_correct = sum(x["correct"] for x in gemini_results)
g_latency = sum(x["latency_ms"] for x in gemini_results) / len(gemini_results)
g_cost_1k = (gemini_total_cost / len(gemini_results)) * 1000

print(f"Gemini 2.5 Flash -> Correct: {g_correct}/{len(gemini_results)} | Avg Latency: {g_latency:.0f} ms | Cost/1k: ${g_cost_1k:.4f}")

# Qwen 8B Summary (based on ~$0.50/hr T4 GPU rental rate)
q_correct = sum(x["correct"] for x in qwen_results)
q_latency = sum(x["latency_ms"] for x in qwen_results) / len(qwen_results)
calls_per_hour = 3600 / (q_latency / 1000)
q_cost_1k = (0.50 / calls_per_hour) * 1000

print(f"Qwen3-8B         -> Correct: {q_correct}/{len(qwen_results)} | Avg Latency: {q_latency:.0f} ms | Cost/1k: ${q_cost_1k:.4f}")

# Save all outputs to file
with open("results.json", "w", encoding="utf-8") as f:
    json.dump({"gemini": gemini_results, "qwen_8b": qwen_results}, f, ensure_ascii=False, indent=2)

Gemini 2.5 Flash -> Correct: 11/15 | Avg Latency: 4131 ms | Cost/1k: $0.3285
Qwen3-8B         -> Correct: 23/60 | Avg Latency: 13765 ms | Cost/1k: $1.9118


In [ ]:
import json
from google.colab import files

# Use whichever variable name your Qwen loop saved to ('qwen_results' or 'qwen_logs')
data_to_save = qwen_results if "qwen_results" in globals() else qwen_logs

with open("qwen_results.json", "w", encoding="utf-8") as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data_to_save)} entries to qwen_results.json!")

# Automatically download the file to your PC
files.download("qwen_results.json")

Saved 60 entries to qwen_results.json!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

eval_suite = [
  {"id": "EN_01", "language": "en", "prompt": "Tomato leaves have dark concentric rings and yellowing around the spots. Identify the disease and recommend a treatment.", "acceptable_answers": ["early blight", "alternaria solani", "alternaria leaf spot", "target spot", "अगेती झुलसा", "अगेती अंगमारी"]},
  {"id": "EN_02", "language": "en", "prompt": "Water-soaked dark lesions on potato leaves with white fungal growth underneath during cool, humid weather. What is this disease?", "acceptable_answers": ["late blight", "phytophthora infestans", "potato blight", "पछेती झुलसा", "पछेती अंगमारी"]},
  {"id": "EN_03", "language": "en", "prompt": "Wheat leaves show orange-brown pustules arranged in parallel stripes. Diagnose the condition and suggest control measures.", "acceptable_answers": ["yellow rust", "stripe rust", "puccinia striiformis", "linear rust", "पीला रतुआ", "धारीदार रतुआ", "पीली गेरुई"]},
  {"id": "EN_04", "language": "en", "prompt": "Diamond-shaped gray lesions with brown borders on paddy leaves are causing neck rot. Identify the rice disease.", "acceptable_answers": ["rice blast", "magnaporthe oryzae", "pyricularia oryzae", "neck blast", "rotten neck", "धान का झोंका", "ब्लास्ट रोग", "गर्दन तोड़"]},
  {"id": "EN_05", "language": "en", "prompt": "Yellow-orange bacterial ooze and wavy yellow margins along rice leaf tips. Which disease is affecting my paddy crop?", "acceptable_answers": ["bacterial leaf blight", "bacterial blight", "blb", "xanthomonas oryzae", "kresek", "जीवाणु झुलसा", "जीवाणु पर्ण अंगमारी"]},
  {"id": "EN_06", "language": "en", "prompt": "White powdery coating on the upper surface of pea and cucurbit leaves, leading to leaf curling. Name the disease.", "acceptable_answers": ["powdery mildew", "erysiphe", "podosphaera", "sphaerotheca", "चूर्णिल आसिता", "छाछ्या", "बुकनी", "सफेद चूर्ण"]},
  {"id": "EN_07", "language": "en", "prompt": "Yellow angular patches on upper grape leaves with purplish-gray fuzz on the underside after rain. Identify the disease.", "acceptable_answers": ["downy mildew", "plasmopara viticola", "मृदुरोमिल आसिता", "डाउनी मिल्ड्यू", "तुलासिता"]},
  {"id": "EN_08", "language": "en", "prompt": "Cotton leaves are curling upward with thickened veins and stunted plant growth. What viral disease is this?", "acceptable_answers": ["cotton leaf curl", "clcv", "clcud", "leaf curl virus", "begomovirus", "पत्ता मरोड़", "लीफ कर्ल", "कुकड़ा"]},
  {"id": "EN_09", "language": "en", "prompt": "Sunken dark circular spots with concentric rings of pink spores on ripe chili fruits. Identify the disease.", "acceptable_answers": ["anthracnose", "die-back", "dieback", "ripe fruit rot", "fruit rot", "colletotrichum", "एन्थ्रेक्नोज", "श्यामव्रण", "फल सड़न"]},
  {"id": "EN_10", "language": "en", "prompt": "Older banana leaves are yellowing from the margins inward and drooping like a skirt around the pseudostem. Diagnose this.", "acceptable_answers": ["panama wilt", "panama disease", "fusarium wilt", "banana wilt", "fusarium oxysporum", "पनामा विल्ट", "पनामा म्लानि", "उकठा"]},
  {"id": "EN_11", "language": "en", "prompt": "Reddish discoloration inside split sugarcane stalks with an alcoholic sour smell. Which disease is this?", "acceptable_answers": ["red rot", "colletotrichum falcatum", "glomerella tucumanensis", "लाल सड़न", "लाल सड़न रोग"]},
  {"id": "EN_12", "language": "en", "prompt": "Cabbage roots are swollen into large distorted galls and plants wilt during the afternoon sun. Identify the disease.", "acceptable_answers": ["clubroot", "club root", "plasmodiophora brassicae", "finger-and-toe", "क्लबरूट", "जड़ गांठ", "मुगदर जड़"]},
  {"id": "EN_13", "language": "en", "prompt": "Soybean leaves show small reddish-brown pustules on the underside and premature defoliation. What disease is this?", "acceptable_answers": ["soybean rust", "asian soybean rust", "phakopsora pachyrhizi", "सोयाबीन रतुआ", "गेरुआ", "रतुआ रोग"]},
  {"id": "EN_14", "language": "en", "prompt": "Black sooty powder replacing grains inside wheat earheads at flowering. Identify this fungal disease.", "acceptable_answers": ["loose smut", "ustilago tritici", "ustilago nuda", "smut", "अनावृत कंडुआ", "कंडुआ", "कंडवा", "लूज स्मट"]},
  {"id": "EN_15", "language": "en", "prompt": "Chickpea plants suddenly droop and dry up while leaves remain green, and the taproot shows dark brown vascular discoloration.", "acceptable_answers": ["fusarium wilt", "chickpea wilt", "vascular wilt", "fusarium oxysporum", "उकठा", "म्लानि", "सूखा रोग"]},
  {"id": "EN_16", "language": "en", "prompt": "Groundnut leaves have dark brown circular spots surrounded by a bright yellow halo. Diagnose the disease.", "acceptable_answers": ["tikka", "cercospora leaf spot", "early leaf spot", "late leaf spot", "mycosphaerella", "टिक्का", "पर्ण चित्ती"]},
  {"id": "EN_17", "language": "en", "prompt": "Mustard leaves and pods show dark brown circular spots with target-like concentric rings. What disease is affecting the crop?", "acceptable_answers": ["alternaria blight", "alternaria leaf spot", "alternaria black spot", "alternaria brassicae", "अल्टरनेरिया झुलसा", "पत्ती धब्बा", "झुलसा रोग"]},
  {"id": "EN_18", "language": "en", "prompt": "Corky raised scabs with yellow halos on citrus leaves, twigs, and fruits. Identify the bacterial infection.", "acceptable_answers": ["citrus canker", "xanthomonas citri", "xanthomonas axonopodis", "canker", "सिट्रस कैंकर", "कैंकर रोग", "कोढ़"]},
  {"id": "EN_19", "language": "en", "prompt": "Mango panicles and young fruits develop black necrotic spots and drop prematurely in humid weather. Name the disease.", "acceptable_answers": ["anthracnose", "mango anthracnose", "blossom blight", "colletotrichum gloeosporioides", "एन्थ्रेक्नोज", "काला धब्बा", "कालवर्ण"]},
  {"id": "EN_20", "language": "en", "prompt": "Maize leaves have long, elliptical grayish-green cigar-shaped lesions on the lower foliage. Identify the disease.", "acceptable_answers": ["turcicum leaf blight", "northern corn leaf blight", "northern corn leaf blotch", "exserohilum turcicum", "helminthosporium turcicum", "पत्ती झुलसा", "झुलसा"]},
  {"id": "EN_21", "language": "en", "prompt": "Okra leaves show prominent yellow-green mosaic patterns along the veins and stunted fruits. What is the diagnosis?", "acceptable_answers": ["yellow vein mosaic", "vein mosaic virus", "yvmv", "bhindi yellow vein", "पीत शिरा मोजेक", "पीला मोजेक"]},
  {"id": "EN_22", "language": "en", "prompt": "Brinjal plants wilt suddenly without yellowing, and milky bacterial ooze appears when the stem is placed in water.", "acceptable_answers": ["bacterial wilt", "ralstonia solanacearum", "pseudomonas solanacearum", "brown rot", "जीवाणु उकठा", "जीवाणु म्लानि", "उकठा"]},
  {"id": "EN_23", "language": "en", "prompt": "Onion leaves show purplish elongated lesions with concentric zones that girdle the leaves and cause them to collapse.", "acceptable_answers": ["purple blotch", "alternaria porri", "बैंगनी धब्बा", "जलेबी रोग"]},
  {"id": "EN_24", "language": "en", "prompt": "Greenish-yellow velvety spore balls replace individual rice grains in the panicle at maturity. Identify the disease.", "acceptable_answers": ["false smut", "ustilaginoidea virens", "villosiclava virens", "green smut", "कंडुआ", "फाल्स स्मट", "मिथ्या कंड", "हल्दी रोग"]},
  {"id": "EN_25", "language": "en", "prompt": "Pearl millet earheads are transformed into leafy green structures instead of normal grain spikes. Name this condition.", "acceptable_answers": ["downy mildew", "green ear", "sclerospora graminicola", "हरित बाली", "हरी बाली", "हरी बालियां", "जोगिया"]},
  {"id": "EN_26", "language": "en", "prompt": "Check today's rainfall forecast and humidity in Nashik before I spray fungicide on my grape vineyard.", "acceptable_answers": ["rainfall", "humidity", "forecast", "weather", "rain", "precipitation"]},
  {"id": "EN_27", "language": "en", "prompt": "Will there be frost or minimum temperatures below 4°C in Karnal over the next 3 days that could damage my wheat crop?", "acceptable_answers": ["frost", "temperature", "forecast", "weather", "4°c", "cold"]},
  {"id": "EN_28", "language": "en", "prompt": "What are the current wind speed and rain chances in Ludhiana so I can plan foliar pesticide spraying?", "acceptable_answers": ["wind speed", "wind", "rain", "forecast", "weather"]},
  {"id": "EN_29", "language": "en", "prompt": "Is heavy rain expected in Indore this week during soybean harvesting?", "acceptable_answers": ["heavy rain", "rain", "rainfall", "forecast", "weather"]},
  {"id": "EN_30", "language": "en", "prompt": "Fetch the 5-day temperature and relative humidity forecast for Shimla to assess apple scab risk.", "acceptable_answers": ["temperature", "humidity", "forecast", "weather", "scab"]},
  {"id": "HI_01", "language": "hi", "prompt": "गेहूं की पत्तियों पर पीले रंग की धारियां दिख रही हैं। यह कौन सा रोग है और इसका क्या उपाय है?", "acceptable_answers": ["yellow rust", "stripe rust", "puccinia striiformis", "पीला रतुआ", "धारीदार रतुआ", "पीली गेरुई", "पीला किट्ट", "रतुआ"]},
  {"id": "HI_02", "language": "hi", "prompt": "टमाटर की पत्तियों पर गहरे भूरे रंग के गोल छल्ले और चारों ओर पीलापन दिख रहा है। रोग की पहचान करें और उपचार बताएं।", "acceptable_answers": ["early blight", "alternaria solani", "अगेती झुलसा", "अगेती अंगमारी", "झुलसा रोग"]},
  {"id": "HI_03", "language": "hi", "prompt": "ठंडे और नम मौसम में आलू की पत्तियों पर काले पानी जैसे धब्बे और निचली सतह पर सफेद फफूंद दिख रही है। यह कौन सा रोग है?", "acceptable_answers": ["late blight", "phytophthora infestans", "पछेती झुलसा", "पछेती अंगमारी", "झुलसा"]},
  {"id": "HI_04", "language": "hi", "prompt": "धान की पत्तियों पर नाव के आकार के भूरे किनारे वाले धब्बे बन रहे हैं और बालियां गर्दन से टूट रही हैं। रोग बताएं।", "acceptable_answers": ["rice blast", "magnaporthe oryzae", "pyricularia oryzae", "झोंका", "ब्लास्ट", "गर्दन तोड़", "बदरा"]},
  {"id": "HI_05", "language": "hi", "prompt": "धान की पत्तियों के किनारों से पीलापन शुरू होकर सूख रहा है और सुबह पत्तियों पर पीली बूंदें दिखती हैं। यह कौन सा रोग है?", "acceptable_answers": ["bacterial leaf blight", "bacterial blight", "xanthomonas oryzae", "जीवाणु झुलसा", "जीवाणु पर्ण अंगमारी", "पर्ण झुलसा"]},
  {"id": "HI_06", "language": "hi", "prompt": "मटर और कद्दू वर्गीय फसलों की पत्तियों की ऊपरी सतह पर सफेद आटे जैसा पाउडर जमा हो गया है। इस रोग का नाम क्या है?", "acceptable_answers": ["powdery mildew", "erysiphe", "चूर्णिल आसिता", "छाछ्या", "बुकनी", "सफेद चूर्ण", "भभूतिया"]},
  {"id": "HI_07", "language": "hi", "prompt": "बारिश के बाद अंगूर की पत्तियों पर पीले धब्बे और निचली सतह पर बैंगनी-भूरे रंग की रुई जैसी फफूंद दिख रही है। रोग की पहचान करें।", "acceptable_answers": ["downy mildew", "plasmopara viticola", "मृदुरोमिल आसिता", "डाउनी मिल्ड्यू", "तुलासिता", "रोमिल फफूंद"]},
  {"id": "HI_08", "language": "hi", "prompt": "कपास की पत्तियां ऊपर की ओर मुड़ रही हैं, नसें मोटी हो गई हैं और पौधे की बढ़वार रुक गई है। यह कौन सा विषाणु रोग है?", "acceptable_answers": ["cotton leaf curl", "clcv", "पत्ता मरोड़", "लीफ कर्ल", "कुकड़ा", "मरोड़िया"]},
  {"id": "HI_09", "language": "hi", "prompt": "पकी हुई मिर्च के फलों पर गहरे धंसे हुए गोल धब्बे और गुलाबी बीजाणुओं के छल्ले बन रहे हैं। इस रोग का क्या नाम है?", "acceptable_answers": ["anthracnose", "die-back", "colletotrichum", "एन्थ्रेक्नोज", "श्यामव्रण", "फल सड़न", "मरोड़िया", "उल्टा सूखा"]},
  {"id": "HI_10", "language": "hi", "prompt": "केले की पुरानी पत्तियां किनारों से पीली पड़कर तने के चारों ओर लटक रही हैं। यह कौन सी बीमारी है?", "acceptable_answers": ["panama wilt", "fusarium wilt", "पनामा विल्ट", "पनामा म्लानि", "उकठा", "म्लानि"]},
  {"id": "HI_11", "language": "hi", "prompt": "गन्ने के तने को बीच से चीरने पर अंदर लाल रंग दिखाई देता है और खट्टी शराब जैसी गंध आती है। यह कौन सा रोग है?", "acceptable_answers": ["red rot", "colletotrichum falcatum", "लाल सड़न", "लाल सड़न रोग"]},
  {"id": "HI_12", "language": "hi", "prompt": "पत्तागोभी की जड़ों में बड़ी-बड़ी गांठें बन गई हैं और दोपहर की धूप में पौधे मुरझा जाते हैं। रोग की पहचान करें।", "acceptable_answers": ["clubroot", "club root", "plasmodiophora", "क्लबरूट", "जड़ गांठ", "मुगदर जड़"]},
  {"id": "HI_13", "language": "hi", "prompt": "सोयाबीन की पत्तियों की निचली सतह पर छोटे लाल-भूरे दाने उभर आए हैं और पत्तियां समय से पहले झड़ रही हैं। रोग बताएं।", "acceptable_answers": ["soybean rust", "phakopsora", "सोयाबीन रतुआ", "गेरुआ", "रतुआ", "किट्ट"]},
  {"id": "HI_14", "language": "hi", "prompt": "गेहूं की बालियों में दानों की जगह काला चूर्ण बन रहा है जो हवा में उड़ जाता है। इस फफूंद रोग का नाम बताएं।", "acceptable_answers": ["loose smut", "ustilago", "अनावृत कंडुआ", "कंडुआ", "कंडवा", "लूज स्मट", "काला कंडुआ"]},
  {"id": "HI_15", "language": "hi", "prompt": "चने के पौधे अचानक मुरझाकर सूख रहे हैं और जड़ को चीरने पर अंदर से गहरी भूरी दिखाई देती है। यह कौन सा रोग है?", "acceptable_answers": ["fusarium wilt", "उकठा", "म्लानि", "सूखा रोग", "विल्ट"]},
  {"id": "HI_16", "language": "hi", "prompt": "मूंगफली की पत्तियों पर गहरे भूरे गोल धब्बे बन गए हैं जिनके चारों ओर पीला घेरा है। इस रोग की पहचान करें।", "acceptable_answers": ["tikka", "cercospora leaf spot", "टिक्का", "पर्ण चित्ती", "पत्ती धब्बा"]},
  {"id": "HI_17", "language": "hi", "prompt": "सरसों की पत्तियों और फलियों पर गहरे भूरे रंग के गोल धब्बे दिख रहे हैं जिनमें चक्र जैसे निशान हैं। यह कौन सा रोग है?", "acceptable_answers": ["alternaria blight", "alternaria leaf spot", "alternaria brassicae", "अल्टरनेरिया झुलसा", "पत्ती धब्बा", "झुलसा"]},
  {"id": "HI_18", "language": "hi", "prompt": "नींबू की पत्तियों, टहनियों और फलों पर खुरदरे उभरे हुए धब्बे हैं जिनके चारों ओर पीला घेरा है। यह कौन सा जीवाणु रोग है?", "acceptable_answers": ["citrus canker", "xanthomonas citri", "सिट्रस कैंकर", "कैंकर", "कोढ़"]},
  {"id": "HI_19", "language": "hi", "prompt": "नमी वाले मौसम में आम के बौर और छोटे फलों पर काले धब्बे पड़ रहे हैं और फल झड़ रहे हैं। रोग का नाम बताएं।", "acceptable_answers": ["anthracnose", "colletotrichum gloeosporioides", "एन्थ्रेक्नोज", "काला धब्बा", "कालवर्ण", "बौर झुलसा"]},
  {"id": "HI_20", "language": "hi", "prompt": "मक्का की निचली पत्तियों पर लंबे, सिगार के आकार के भूरे-हरे धब्बे दिखाई दे रहे हैं। रोग की पहचान करें।", "acceptable_answers": ["turcicum leaf blight", "northern corn leaf blight", "exserohilum turcicum", "पत्ती झुलसा", "पर्ण अंगमारी", "झुलसा"]},
  {"id": "HI_21", "language": "hi", "prompt": "भिंडी की पत्तियों की नसें पीली पड़ गई हैं और फल छोटे व टेढ़े-मेढ़े रह गए हैं। यह कौन सा रोग है?", "acceptable_answers": ["yellow vein mosaic", "yvmv", "पीत शिरा मोजेक", "पीला मोजेक", "नस पीली"]},
  {"id": "HI_22", "language": "hi", "prompt": "बैंगन के हरे पौधे अचानक मुरझा रहे हैं और तने को काटकर पानी में डालने पर सफेद दूधिया पदार्थ निकलता है। रोग बताएं।", "acceptable_answers": ["bacterial wilt", "ralstonia solanacearum", "pseudomonas solanacearum", "जीवाणु उकठा", "जीवाणु म्लानि", "उकठा"]},
  {"id": "HI_23", "language": "hi", "prompt": "प्याज की पत्तियों पर बैंगनी रंग के लंबे धब्बे पड़ रहे हैं जिससे पत्तियां बीच से टूटकर गिर रही हैं। इस रोग का नाम क्या है?", "acceptable_answers": ["purple blotch", "alternaria porri", "बैंगनी धब्बा", "जलेबी रोग"]},
  {"id": "HI_24", "language": "hi", "prompt": "धान की बालियों में पकने के समय दानों की जगह हरे-पीले मखमली गोल पिंड बन रहे हैं। इस रोग की पहचान करें।", "acceptable_answers": ["false smut", "ustilaginoidea virens", "कंडुआ", "फाल्स स्मट", "मिथ्या कंड", "हल्दी रोग", "कंडवा"]},
  {"id": "HI_25", "language": "hi", "prompt": "बाजरे की बालियों में दानों की जगह हरी पत्तियों जैसी संरचना बन गई है। इस रोग को क्या कहते हैं?", "acceptable_answers": ["downy mildew", "green ear", "sclerospora graminicola", "हरित बाली", "हरी बाली", "हरी बालियां", "जोगिया"]},
  {"id": "HI_26", "language": "hi", "prompt": "अंगूर के बाग में फफूंदनाशक छिड़कने से पहले नासिक में आज की बारिश और नमी का पूर्वानुमान बताएं।", "acceptable_answers": ["बारिश", "नमी", "पूर्वानुमान", "मौसम", "वर्षा", "आर्द्रता"]},
  {"id": "HI_27", "language": "hi", "prompt": "क्या अगले 3 दिनों में करनाल में पाला पड़ने या तापमान 4 डिग्री सेल्सियस से नीचे जाने की संभावना है?", "acceptable_answers": ["पाला", "तापमान", "मौसम", "4 डिग्री", "शीत लहर", "ठंड"]},
  {"id": "HI_28", "language": "hi", "prompt": "कीटनाशक छिड़काव की योजना बनाने के लिए लुधियाना में अभी हवा की गति और बारिश की संभावना क्या है?", "acceptable_answers": ["हवा की गति", "हवा", "बारिश", "मौसम", "पूर्वानुमान", "वर्षा"]},
  {"id": "HI_29", "language": "hi", "prompt": "क्या इस सप्ताह सोयाबीन की कटाई के दौरान इंदौर में भारी बारिश होने की संभावना है?", "acceptable_answers": ["बारिश", "मौसम", "पूर्वानुमान", "वर्षा"]},
  {"id": "HI_30", "language": "hi", "prompt": "सेब में स्कैब रोग के जोखिम का आकलन करने के लिए शिमला का अगले 5 दिनों का तापमान और आर्द्रता पूर्वानुमान बताएं।", "acceptable_answers": ["तापमान", "आर्द्रता", "पूर्वानुमान", "मौसम", "नमी", "स्कैब"]}
]

# Save clean JSON to disk
with open("eval_suite.json", "w", encoding="utf-8") as f:
    json.dump(eval_suite, f, ensure_ascii=False, indent=2)

# Re-score your existing qwen_results.json immediately
ground_truth = {item["id"]: item["acceptable_answers"] for item in eval_suite}

with open("qwen_results.json", "r", encoding="utf-8") as f:
    rows = json.load(f)

en_ok, hi_ok = 0, 0
for r in rows:
    passed = any(term.lower() in r["output"].lower() for term in ground_truth[r["id"]])
    r["correct"] = passed
    if r["id"].startswith("EN_") and passed:
        en_ok += 1
    elif r["id"].startswith("HI_") and passed:
        hi_ok += 1

with open("qwen_results.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

avg_ms = sum(r["latency_ms"] for r in rows) / len(rows)
cost_1k = (0.50 / (3600 / (avg_ms / 1000))) * 1000

print(f"Saved clean eval_suite.json ({len(eval_suite)} items)")
print(f"Qwen3-8B -> EN: {en_ok}/30 | HI: {hi_ok}/30 | Total: {en_ok+hi_ok}/60 | Avg Latency: {avg_ms:.0f} ms | Cost/1k: ${cost_1k:.4f}")

Saved clean eval_suite.json (60 items)
Qwen3-8B -> EN: 22/30 | HI: 10/30 | Total: 32/60 | Avg Latency: 13765 ms | Cost/1k: $1.9118
